# AlphaLabLite Project Walkthrough

**AlphaLabLite** is a lightweight Python project that executes a small domain-specific language (DSL) for time-series transformation workflows. This notebook is designed as an interview/demo walkthrough: it explains the motivation, architecture, files, execution pipeline, CLI, API, SQLite storage, and testing strategy.

## 1. Overview

AlphaLabLite lets a user write simple scripts such as:

```text
price = Fetch{OneMinuteGoldPrices}{}
fast = ExponentialMovingAverage{0.3}{price}
slow = SimpleMovingAverage{20}{price}
entry = CrossAbove{}{fast, slow}
exit = CrossAbove{}{slow, fast}
result = PortfolioSimulation{10000}{entry, exit, price}
```

The project parses each line, executes the correct Python function, stores intermediate variables in memory, saves the final execution state to SQLite, and exposes the workflow through both a CLI and a REST API.

## 2. Problem Statement

Many small data workflows follow a repeated pattern:

1. Load a dataset or time series.
2. Apply transformations.
3. Generate signals.
4. Simulate or calculate an output.
5. Save the result.
6. View selected outputs later.

AlphaLabLite demonstrates this workflow using a compact DSL. Instead of requiring the user to manually call Python functions, the user describes the workflow as readable script lines.

## 3. System Design

```text
Script
  ↓
Parser
  ↓
Parsed Commands
  ↓
Engine
  ↓
Transformation Execution
  ↓
Memory
  ↓
SQLite
  ↓
CLI / REST API
```

The design separates responsibilities clearly:

- `parsing.py` understands the DSL syntax.
- `engine.py` executes parsed commands.
- `transformations.py` contains calculation logic.
- `storage.py` persists results.
- `main.py` exposes a CLI.
- `app.py` exposes a REST API.

## 4. Full Pipeline

A full script run follows this pipeline:

```text
User script
  ↓
parseScript(script)
  ↓
List of command dictionaries
  ↓
executeCommand(command, memory)
  ↓
Transformation function call
  ↓
Updated memory dictionary
  ↓
save_result(memory)
  ↓
SQLite rows grouped by script_id
  ↓
CLI output or API JSON response
```

The key idea is that each command can depend on variables created by earlier commands.

## 5. `transformations.py`

### Role

`transformations.py` is the calculation layer. It contains reusable functions for fetching sample data, calculating indicators, generating signals, and simulating a simple portfolio.

### Inputs

- Datasource labels such as `OneMinuteGoldPrices`
- Time-series lists
- Config values such as moving average window, EMA alpha, period, or starting balance

### Outputs

- New time-series lists
- Signal lists containing `0` and `1`
- Portfolio value lists

### Core Logic

Important functions include:

- `Fetch(datasource)`: loads a row from `data/fetch_transformation_data.csv`.
- `simpleMovingAverage(A, window)`: calculates a rolling average.
- `ExponentialMovingAverage(alpha, A)`: calculates EMA values.
- `RateOfChange(A, period)`: calculates relative change over a period.
- `CrossAbove(A1, A2)`: detects crossover signals.
- `ConstantSeries(A, k)`: creates a constant list with the same length as `A`.
- `PortfolioSimulation(balance, price, entry, exit)`: simulates a simple buy/sell strategy.

### Interaction With Other Modules

`engine.py` imports this file and maps DSL function names to these Python functions.

### Small Example

```text
slow = SimpleMovingAverage{20}{price}
```

The engine resolves `price` from memory and calls the moving average function with a window of `20`.

## 6. `parsing.py`

### Role

`parsing.py` converts DSL script text into structured Python dictionaries.

### Inputs

- A single command line
- Or a full multi-line script

### Outputs

A parsed command dictionary:

```python
{
    "variable": "fast",
    "function": "ExponentialMovingAverage",
    "configs": ["0.3"],
    "inputs": ["price"]
}
```

### Core Logic

- `splitArguments(arguments)` splits comma-separated values.
- `parseCommand(command)` validates and parses one DSL line.
- `parseScript(script)` parses a full script line by line.

### Interaction With Other Modules

`engine.py` calls `parseScript` before execution.

### Small Example

```text
price = Fetch{OneMinuteGoldPrices}{}
```

becomes a dictionary with variable `price`, function `Fetch`, config `OneMinuteGoldPrices`, and no inputs.

## 7. `engine.py`

### Role

`engine.py` is the coordinator. It receives a script, parses it, executes commands in order, and stores outputs in memory.

### Inputs

- Raw script text
- Parsed command dictionaries
- Memory values produced by earlier commands

### Outputs

- A final memory dictionary containing every variable created during the script

### Core Logic

- `TRANSFORMATIONS` maps DSL names to real Python functions.
- `configToArgs` converts configs like `20` or `0.3` into numeric types.
- `executeCommand` runs one command and updates memory.
- `executeScript` runs the full script from top to bottom.

### Interaction With Other Modules

`engine.py` depends on:

- `parsing.py` for parsing
- `transformations.py` for actual calculations

It is used by:

- `main.py` for CLI execution
- `app.py` for API execution

### Small Example

If memory contains `price`, this command:

```text
fast = ExponentialMovingAverage{0.3}{price}
```

results in a new memory entry named `fast`.

## 8. `storage.py`

### Role

`storage.py` persists execution results using SQLite.

### Inputs

- A memory dictionary after script execution
- A `script_id`
- A list of variable names to retrieve

### Outputs

- A generated `script_id` after saving
- A dictionary of loaded variable values when viewing results

### Core Logic

- `init_db()` creates the table if needed.
- `save_result(memory)` saves each memory variable as JSON under one script ID.
- `load_result(script_id, variable_names)` retrieves selected variables.

### SQLite Table

```text
alpha_lab_lite
├── id
├── script_id
├── variable_name
└── variable_json
```

### Small Example

After running a script, `save_result(memory)` stores variables like `price`, `fast`, `slow`, and `result`. Later, the CLI or API can load only the requested variables.

## 9. `main.py`

### Role

`main.py` provides the command-line interface.

### Inputs

- Terminal command arguments
- Script file path
- Script ID and variable names for viewing results

### Outputs

- Printed `script_id` after execution
- Printed variable values when viewing saved results

### Core Logic

- Uses `argparse` to define `execute` and `view` commands.
- Reads a script file for execution.
- Calls the engine and storage functions.

### Small Example

```bash
python main.py execute script.txt
```

Then:

```bash
python main.py view --id YOUR_SCRIPT_ID price result
```

## 10. `app.py`

### Role

`app.py` exposes AlphaLabLite through a FastAPI REST API.

### Inputs

- JSON body containing script text for `/execute`
- Query parameters for `/view`

### Outputs

- JSON response containing a success message and `script_id`
- JSON response containing requested variable values

### Core Logic

- `POST /execute` runs a script and saves outputs.
- `GET /view` retrieves saved variables.

### Small Example

```text
POST /execute
{
  "script": "price = Fetch{OneMinuteGoldPrices}{}"
}
```

Response:

```json
{
  "message": "Script successfully executed",
  "script_id": "generated-script-id"
}
```

## 11. CLI Demo

From the project root, run:

```bash
python main.py execute script.txt
```

The program prints a `script_id`.

Then retrieve selected variables:

```bash
python main.py view --id YOUR_SCRIPT_ID price result
```

This demonstrates the full local pipeline: script file → engine → SQLite → terminal output.

## 12. REST API Demo

Start the API server:

```bash
uvicorn app:app --reload
```

Open the interactive docs:

```text
http://127.0.0.1:8000/docs
```

Execute a script using `POST /execute`, then retrieve variables using `GET /view`.

Example view URL:

```text
http://127.0.0.1:8000/view?script_id=YOUR_SCRIPT_ID&variables=price,result
```

## 13. SQLite Persistence

AlphaLabLite stores each script execution under a unique UUID called `script_id`.

Each variable is saved as one SQLite row:

```text
script_id       variable_name       variable_json
------------------------------------------------
abc-123         price               [100, 101, ...]
abc-123         fast                [100, 100.3, ...]
abc-123         result              [10000, ...]
```

This makes it simple to retrieve only the variables needed for a demo or analysis.

## 14. Testing

A practical testing strategy for this project would include:

### Parser Tests

- Valid command parsing
- Empty line handling
- Invalid command errors

### Transformation Tests

- Moving average expected values
- EMA expected values
- Rate of change with normal and zero values
- CrossAbove signal behavior
- Portfolio simulation output

### Engine Tests

- Full script execution
- Unknown function handling
- Variable used before creation

### Storage Tests

- Save memory to SQLite
- Load selected variables
- Missing variable returns `None`

### Manual Smoke Test

```bash
python main.py execute script.txt
python main.py view --id YOUR_SCRIPT_ID price result
```

## 15. Lessons Learned

This project demonstrates several useful software engineering ideas:

- How to design a small DSL with a consistent syntax.
- How to separate parsing from execution.
- How to map user-facing function names to real Python functions.
- How to manage state with an in-memory dictionary.
- How to persist structured outputs using SQLite and JSON.
- How to expose the same core logic through both a CLI and an API.

The project is intentionally compact, which makes it easier to explain and maintain.

## Optional Helper Visualization

The project PDF does not require plotting. This small visualization section is only a helper for the demo so the time-series outputs are easier to understand. It is not part of the core engine, CLI, API, or storage requirements.

The code below runs a simple example script and displays small line charts for `price`, `momentum`, `entry`, `exit`, and `result`. It uses the in-memory output from the engine so it stays simple and easy to run inside the notebook.

In [ ]:
from html import escape
import math

from IPython.display import HTML, display

from engine import executeScript

demo_script = """
price = Fetch{OneMinuteGoldPrices}{}
momentum = RateOfChange{3}{price}
zero = ConstantSeries{0}{price}
entry = CrossAbove{}{momentum, zero}
exit = CrossAbove{}{zero, momentum}
result = PortfolioSimulation{10000}{entry, exit, price}
"""

data = executeScript(demo_script)
variables = ["price", "momentum", "entry", "exit", "result"]

def clean_number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(number):
        return None
    return number

def svg_line_chart(values, width=680, height=180):
    numbers = [clean_number(value) for value in values]
    valid_values = [value for value in numbers if value is not None]
    if not valid_values:
        return "<p>No numeric values to plot.</p>"

    min_value = min(valid_values)
    max_value = max(valid_values)
    value_range = max_value - min_value or 1
    padding = 26
    chart_width = width - padding * 2
    chart_height = height - padding * 2
    points = []

    for index, value in enumerate(numbers):
        if value is None:
            continue
        x = padding + (index / max(len(numbers) - 1, 1)) * chart_width
        y = padding + (max_value - value) / value_range * chart_height
        points.append(f"{x:.2f},{y:.2f}")

    return f"""
    <svg width="{width}" height="{height}" viewBox="0 0 {width} {height}">
        <rect width="{width}" height="{height}" fill="#ffffff"/>
        <line x1="{padding}" y1="{height - padding}" x2="{width - padding}" y2="{height - padding}" stroke="#9ca3af"/>
        <line x1="{padding}" y1="{padding}" x2="{padding}" y2="{height - padding}" stroke="#9ca3af"/>
        <polyline points="{' '.join(points)}" fill="none" stroke="#2563eb" stroke-width="2.5"/>
        <text x="{padding}" y="18" font-size="12" fill="#374151">max: {max_value:.4g}</text>
        <text x="{padding}" y="{height - 6}" font-size="12" fill="#374151">min: {min_value:.4g}</text>
    </svg>
    """

sections = []
for name in variables:
    values = data[name]
    sections.append(f"<h3>{escape(name)}</h3>{svg_line_chart(values)}")

display(HTML("<h2>Demo Charts</h2>" + "".join(sections)))
print("Demo visualization created from engine output.")


## 16. Final Summary

AlphaLabLite is a clean demonstration of a script-driven time-series execution pipeline. The user writes commands in a small DSL, the parser converts them into structured commands, the engine executes the correct transformation functions, memory stores intermediate results, SQLite persists outputs, and the CLI/API provide two ways to interact with the system.

### One-Minute Interview Explanation

AlphaLabLite is a small Python DSL engine for time-series workflows. A user writes a script using the format `variable = Function{configs}{inputs}`. The parser turns each line into structured commands, the engine resolves dependencies from memory, runs transformation functions such as moving averages and crossover signals, and saves the generated variables into SQLite. I exposed the same execution flow through both a command-line interface and a FastAPI API. The project demonstrates parsing, function dispatch, state management, persistence, and API design in a compact and easy-to-explain system.